In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.data_script import *

### Load Data

In [2]:
# Import datasets from
# lmarena-ai/VisionArena-Battle
# https://huggingface.co/datasets/lmsys
from datasets import load_dataset
ds = load_dataset("lmarena-ai/webdev-arena-preference-10k")

/Users/JennyH/Library/Python/3.8/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# inspect the available splits
print(ds)  
# grab the test split
test = ds["test"]

DatasetDict({
    test: Dataset({
        features: ['model_a', 'model_b', 'conversation_a', 'conversation_b', 'winner', 'tstamp', 'anony', 'question_id'],
        num_rows: 10501
    })
})


In [8]:
df = test.to_pandas()
df.shape

(10501, 8)

In [9]:
df.head()

,model_a,model_b,conversation_a,conversation_b,winner,tstamp,anony,question_id
0,claude-3-5-sonnet-20241022,gemini-2.0-flash-thinking-exp-1219,"[{'content': [{'text': 'Machine,\nPls make web...","[{'content': [{'text': 'Machine,\nPls make web...",model_a,1.734980e+09,True,38fa1506bce8828d1996156a0a422bee
1,qwen-2.5-coder-32b-instruct,claude-3-5-sonnet-20241022,[{'content': [{'text': 'portfolio page for a f...,[{'content': [{'text': 'portfolio page for a f...,model_b,1.734879e+09,True,6d09f8ab32c62f355eb30a45192e485b
2,gemini-1.5-pro-002,qwen-2.5-coder-32b-instruct,"[{'content': [{'text': 'Snake Game', 'type': '...","[{'content': [{'text': 'Snake Game', 'type': '...",tie,1.734376e+09,True,d3c79f8667add0d2d1e33eba25dd6cec
3,gpt-4o-2024-11-20,claude-3-5-sonnet-20241022,[{'content': [{'text': 'Generate me a UI for d...,[{'content': [{'text': 'Generate me a UI for d...,model_b,1.734081e+09,True,9f688abafa4b68a72d63a46985854576
4,gemini-2.0-flash-thinking-exp-1219,claude-3-5-sonnet-20241022,"[{'content': [{'text': ""Build me Elon Musk's T...","[{'content': [{'text': ""Build me Elon Musk's T...",model_b,1.735158e+09,True,29fd8cb9774c3a6fa74af5d38e597123


In [10]:
# create a column winner_model_a, which is 1 if model_a is preferred, 0 if model_b is preferred
df['winner_model_a'] = df['winner'].apply(lambda x: 1 if x == 'model_a' else 0)
# create a column called winner_tie that is 1 if the winner is 'tie', else 0
df['winner_tie'] = df['winner'].apply(lambda x: 1 if x == 'tie' else 0)
df.head()

,model_a,model_b,conversation_a,conversation_b,winner,tstamp,anony,question_id,winner_model_a,winner_tie
0,claude-3-5-sonnet-20241022,gemini-2.0-flash-thinking-exp-1219,"[{'content': [{'text': 'Machine,\nPls make web...","[{'content': [{'text': 'Machine,\nPls make web...",model_a,1.734980e+09,True,38fa1506bce8828d1996156a0a422bee,1,0
1,qwen-2.5-coder-32b-instruct,claude-3-5-sonnet-20241022,[{'content': [{'text': 'portfolio page for a f...,[{'content': [{'text': 'portfolio page for a f...,model_b,1.734879e+09,True,6d09f8ab32c62f355eb30a45192e485b,0,0
2,gemini-1.5-pro-002,qwen-2.5-coder-32b-instruct,"[{'content': [{'text': 'Snake Game', 'type': '...","[{'content': [{'text': 'Snake Game', 'type': '...",tie,1.734376e+09,True,d3c79f8667add0d2d1e33eba25dd6cec,0,1
3,gpt-4o-2024-11-20,claude-3-5-sonnet-20241022,[{'content': [{'text': 'Generate me a UI for d...,[{'content': [{'text': 'Generate me a UI for d...,model_b,1.734081e+09,True,9f688abafa4b68a72d63a46985854576,0,0
4,gemini-2.0-flash-thinking-exp-1219,claude-3-5-sonnet-20241022,"[{'content': [{'text': ""Build me Elon Musk's T...","[{'content': [{'text': ""Build me Elon Musk's T...",model_b,1.735158e+09,True,29fd8cb9774c3a6fa74af5d38e597123,0,0


In [11]:
ties = df[df['winner_tie'] == 1]
print(f"Number of ties: {len(ties)}")
# proportion of ties.
print(f"Proportion of ties: {len(ties) / len(df):.2%}")
# note, the proportion of ties is 9.17% for the LLM-as-judge data and 23.25% for the human-as-judge data.

Number of ties: 766
Proportion of ties: 7.29%


In [12]:
rawBT = df[['model_a', 'model_b', 'winner_model_a', 'winner_tie']]
rawBT.head()
rawBT.shape
# rawBT_noTies.head() # (2575, 3)
# rawBT_noTies.shape

(10501, 4)

In [13]:
# how to get the unique names in both columns
model_a_names = df['model_a'].unique()
model_b_names = df['model_b'].unique()
# combine the two arrays and get the unique names
model_names = np.unique(np.concatenate((model_a_names, model_b_names)))
# print the number of unique model names
print(f"Number of unique model names: {len(model_names)}")

Number of unique model names: 13


In [14]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

claude-3-5-haiku-20241022: 1119
claude-3-5-sonnet-20241022: 2637
deepseek-v3: 969
gemini-1.5-pro-002: 2265
gemini-2.0-flash-exp: 2319
gemini-2.0-flash-thinking-exp-1219: 1677
gemini-exp-1206: 2437
gpt-4o-2024-11-20: 2523
llama-v3.1-405b-instruct: 235
llama-v3.3-70b-instruct: 25
o1-2024-12-17: 790
o1-mini-2024-09-12: 1746
qwen-2.5-coder-32b-instruct: 2260


In [15]:
# make the BT design matrix.
X, y, player_to_id = make_BT_design_matrix(rawBT, weight_tie = True)
X.shape, y.shape

((21002, 12), (21002,))

#### Run Top-k Robustness Check.

In [16]:
ks = [1, 5]
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = True)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [17]:
# find the (k, alpha N) pairs that are non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 18): (11,
  None,
  -2.6720424681191832,
  0.1837221334672124,
  array([ 7164,  7539,  9112,  7711,  2089,  1815,  2414,  6542,  6446,
          4883,  8753,  2889,  9272,  3553,  1512,  5933,  6992, 10387])),
 (5, 13): (3,
  11,
  1.2819651846202045,
  -0.12369178206385278,
  array([7164, 7539, 9112, 7711, 1815, 2089, 2414, 6542, 6446, 4883, 8753,
         2889, 9272]))}

In [ ]:
# save results as a .pkl file.
# import pickle
# with open('results/webdevNonrobustWtd.pkl', 'wb') as f:
#     pickle.dump(results_nonrobust, f)

In [22]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

claude-3-5-haiku-20241022: 1119
claude-3-5-sonnet-20241022: 2637
deepseek-v3: 969
gemini-1.5-pro-002: 2265
gemini-2.0-flash-exp: 2319
gemini-2.0-flash-thinking-exp-1219: 1677
gemini-exp-1206: 2437
gpt-4o-2024-11-20: 2523
llama-v3.1-405b-instruct: 235
llama-v3.3-70b-instruct: 25
o1-2024-12-17: 790
o1-mini-2024-09-12: 1746
qwen-2.5-coder-32b-instruct: 2260


In [18]:
from package.RankAMIP.plot_util import *
rankings = return_rankings_list(X, y, results, 1, 18, player_to_id)

In [19]:
# plot the rankings on the original arena
filename_to_save = 'fig/top10_webdev.png'
plot_title = 'Model Rankings in Webdev Arena'
plot_bt_scores(X, y, rankings, alphaN, 10, plot_title, filename_to_save)

In [21]:
results_nonrobust

{(1, 18): (11,
  None,
  -2.6720424681191832,
  0.1837221334672124,
  array([ 7164,  7539,  9112,  7711,  2089,  1815,  2414,  6542,  6446,
          4883,  8753,  2889,  9272,  3553,  1512,  5933,  6992, 10387])),
 (5, 13): (3,
  11,
  1.2819651846202045,
  -0.12369178206385278,
  array([7164, 7539, 9112, 7711, 1815, 2089, 2414, 6542, 6446, 4883, 8753,
         2889, 9272]))}

In [23]:
# find the two models that changed ranking.
## Count number of games between the first- and second-place models played in total.
is_haiku_sonnet = (
    (df['model_a'].str.contains('claude-3-5-sonnet-20241022') & df['model_b'].str.contains('claude-3-5-haiku-20241022')) |
    (df['model_a'].str.contains('claude-3-5-haiku-20241022') & df['model_b'].str.contains('claude-3-5-sonnet-20241022'))
)

num_haiku_sonnet = df[is_haiku_sonnet].shape[0]
print("Number of games between claude haiku and claude sonnet: ", num_haiku_sonnet)

Number of games between claude haiku and claude sonnet:  95


In [24]:
# model pairs
df['model_pair'] = df.apply(lambda row: tuple(sorted([row['model_a'], row['model_b']])), axis=1)
df['model_pair']

# Compute average games per model pair across the arena.
pair_counts = df['model_pair'].value_counts()
average_games_per_pair = pair_counts.mean()
print("Average number of games per model pair:", average_games_per_pair)

Average number of games per model pair: 145.84722222222223


In [30]:
df['winner_model_b'] = ((df['winner_model_a'] == 0) & (df['winner_tie'] == 0)).astype(int)
df.head()

,model_a,model_b,conversation_a,conversation_b,winner,tstamp,anony,question_id,winner_model_a,winner_tie,model_pair,winner_model_b
0,claude-3-5-sonnet-20241022,gemini-2.0-flash-thinking-exp-1219,"[{'content': [{'text': 'Machine,\nPls make web...","[{'content': [{'text': 'Machine,\nPls make web...",model_a,1.734980e+09,True,38fa1506bce8828d1996156a0a422bee,1,0,"(claude-3-5-sonnet-20241022, gemini-2.0-flash-...",0
1,qwen-2.5-coder-32b-instruct,claude-3-5-sonnet-20241022,[{'content': [{'text': 'portfolio page for a f...,[{'content': [{'text': 'portfolio page for a f...,model_b,1.734879e+09,True,6d09f8ab32c62f355eb30a45192e485b,0,0,"(claude-3-5-sonnet-20241022, qwen-2.5-coder-32...",1
2,gemini-1.5-pro-002,qwen-2.5-coder-32b-instruct,"[{'content': [{'text': 'Snake Game', 'type': '...","[{'content': [{'text': 'Snake Game', 'type': '...",tie,1.734376e+09,True,d3c79f8667add0d2d1e33eba25dd6cec,0,1,"(gemini-1.5-pro-002, qwen-2.5-coder-32b-instruct)",0
3,gpt-4o-2024-11-20,claude-3-5-sonnet-20241022,[{'content': [{'text': 'Generate me a UI for d...,[{'content': [{'text': 'Generate me a UI for d...,model_b,1.734081e+09,True,9f688abafa4b68a72d63a46985854576,0,0,"(claude-3-5-sonnet-20241022, gpt-4o-2024-11-20)",1
4,gemini-2.0-flash-thinking-exp-1219,claude-3-5-sonnet-20241022,"[{'content': [{'text': ""Build me Elon Musk's T...","[{'content': [{'text': ""Build me Elon Musk's T...",model_b,1.735158e+09,True,29fd8cb9774c3a6fa74af5d38e597123,0,0,"(claude-3-5-sonnet-20241022, gemini-2.0-flash-...",1


In [33]:
# Find the win margin between 'claude-3-5-haiku-20241022' and 'claude-3-5-sonnet-20241022'
# that is, find all games that are between the two models.
dfFlippedRanking = df[is_haiku_sonnet]
## Count number of games between that 'claude-3-5-haiku-20241022' won.
haiku_wins = (
    (dfFlippedRanking['model_a'].str.contains('claude-3-5-haiku-20241022') & dfFlippedRanking['winner_model_a'] == 1) |
    (dfFlippedRanking['model_b'].str.contains('claude-3-5-haiku-20241022') & dfFlippedRanking['winner_model_b'] == 1)
)
num_haiku_wins = dfFlippedRanking[haiku_wins].shape[0]
 # 0.5373134328358209

sonnet_wins = (
    (dfFlippedRanking['model_a'].str.contains('claude-3-5-sonnet-20241022') & dfFlippedRanking['winner_model_a'] == 1) |
    (dfFlippedRanking['model_b'].str.contains('claude-3-5-sonnet-20241022') & dfFlippedRanking['winner_model_b'] == 1)
)

num_haiku_wins = dfFlippedRanking[haiku_wins].shape[0]
num_sonnet_wins = dfFlippedRanking[sonnet_wins].shape[0]

print("Proportion of games that claude sonnet won: ", num_sonnet_wins / (num_haiku_wins + num_sonnet_wins))


Proportion of games that claude sonnet won:  0.6976744186046512


In [32]:
num_haiku_wins, num_sonnet_wins

(26, 60)